# Búsqueda de Hiperparámetros para CNN en MRI
Este notebook contiene únicamente la lógica de búsqueda de hiperparámetros para el modelo CNN

In [ ]:
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
import itertools
from PIL import Image
from sklearn.model_selection import train_test_split

## Carga y preparación de datos
Se cargan las imágenes preprocesadas, se normalizan y se dividen en conjuntos de entrenamiento, validación y test para la búsqueda de hiperparámetros.

In [ ]:
def cargar_datos_preprocesados(ruta_base, clases, img_size=(224, 224)):
    X, y = [], []
    for idx, clase in enumerate(clases):
        ruta_clase = os.path.join(ruta_base, clase)
        for nombre_img in os.listdir(ruta_clase):
            ruta_img = os.path.join(ruta_clase, nombre_img)
            try:
                img = Image.open(ruta_img).convert('L')
                img = img.resize(img_size)
                X.append(np.array(img))
                y.append(idx)
            except Exception as e:
                print(f"Error cargando {ruta_img}: {e}")
    X = np.array(X)
    y = np.array(y)
    return X, y

# Definir rutas y clases
RUTA_PREPROCESADAS = '../data/preprocesadas'
CLASES = ['brain_glioma', 'brain_menin', 'brain_tumor']
IMG_SIZE = (224, 224)

# Cargar datos
X, y = cargar_datos_preprocesados(RUTA_PREPROCESADAS, CLASES, IMG_SIZE)
print(f"Total de imágenes cargadas: {X.shape[0]}")
print(f"Dimensiones de las imágenes: {X.shape[1:]} (esperado: {IMG_SIZE})")
print(f"Distribución de clases: {np.bincount(y)}")
# Normalizar y expandir dimensiones para CNN (canal único)
X = X.astype('float32') / 255.0
X = np.expand_dims(X, axis=-1)

y_cat = to_categorical(y, num_classes=len(CLASES))

# División: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y_cat, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=np.argmax(y_temp, axis=1), random_state=42)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

## Definición de la arquitectura base de la CNN
Se define una función para crear el modelo CNN parametrizable según los hiperparámetros a explorar.

In [ ]:
def crear_modelo(img_size=(224,224), n_clases=3, conv_filters=[32, 64, 128], dense_units=128, dropout_conv=0.2, dropout_dense=0.4, lr=1e-3):
    model = Sequential()
    model.add(Conv2D(conv_filters[0], (3,3), activation='relu', input_shape=(img_size[0], img_size[1], 1)))
    model.add(MaxPooling2D((2,2)))
    model.add(Dropout(dropout_conv))
    model.add(Conv2D(conv_filters[1], (3,3), activation='relu'))
    model.add(MaxPooling2D((2,2)))
    model.add(Dropout(dropout_conv))
    model.add(Conv2D(conv_filters[2], (3,3), activation='relu'))
    model.add(MaxPooling2D((2,2)))
    model.add(Dropout(dropout_conv + 0.1))
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_dense))
    model.add(Dense(n_clases, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## Definición del espacio de hiperparámetros
Se define el grid de hiperparámetros y una función para expandir todas las combinaciones posibles a explorar.

In [ ]:
param_grid = [
    {'conv_filters': [[32, 64, 128]], 'dense_units': [128], 'dropout_conv': [0.2], 'dropout_dense': [0.4], 'lr': [1e-3]},
    {'conv_filters': [[16, 32, 64]], 'dense_units': [64], 'dropout_conv': [0.1], 'dropout_dense': [0.3], 'lr': [5e-4]},
    {'conv_filters': [[64, 128, 256]], 'dense_units': [256], 'dropout_conv': [0.3], 'dropout_dense': [0.5], 'lr': [1e-4]},
]

def expand_grid(param_grid):
    for grid in param_grid:
        keys, values = zip(*grid.items())
        for v in itertools.product(*values):
            yield dict(zip(keys, v))

## Búsqueda de hiperparámetros y selección del mejor modelo
Se entrena la CNN con cada combinación de hiperparámetros y se selecciona la mejor configuración según la precisión de validación.

In [ ]:
results = []
best_val_acc = 0
best_params = None
best_model = None

for i, params in enumerate(expand_grid(param_grid)):
    print(f'[INFO] Entrenando configuración {i+1}: {params}')
    model = crear_modelo(img_size=(224,224), n_clases=3, **params)
    early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )
    val_acc = max(history.history['val_accuracy'])
    print(f'Precisión de validación máxima: {val_acc:.4f}')
    results.append({'params': params, 'val_acc': val_acc, 'history': history})
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = params
        best_model = model

print('[RESULTADO] Mejor configuración encontrada:')
print(best_params)
print(f'Precisión de validación: {best_val_acc:.4f}')